# Phase 3: PlantVillage and PlantDoc preliminary image audit

This notebook accompanies `scripts/analyze_data.py`. It reports **measured** image counts, class imbalance, dimensions, duplicate/related-leaf candidates, and reproducible examples for visual inspection. The original dataset category names are retained; category matching is a separate task. No model is trained, and the raw images are never modified.

The screenshots and charts in the assignment are examples, **not results**. Execute these cells against the prepared local datasets before drawing conclusions. The sample is not a substitute for checking every image or validating disease labels.


## 1. Run the inventory

Run this once after `python scripts/prepare_data.py`; rerun if the underlying files change. Set `DATA_ROOT` if using another machine. The script records source versions from the local manifest and scans *all supported image files*, but computes its sharpness proxy only on a seeded sample of up to four images per dataset/category.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "analyze_data.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # notebook opened from notebooks/
DATA_ROOT = Path.home() / "datasets" / "cecs551-plant-disease"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "analyze_data.py"),
                "--data-root", str(DATA_ROOT), "--output-dir", str(OUTPUT_DIR),
                "--samples-per-category", "4", "--seed", "551"], check=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

inventory = pd.read_csv(OUTPUT_DIR / "image_inventory.csv", keep_default_na=False)
counts = pd.read_csv(OUTPUT_DIR / "class_counts.csv", keep_default_na=False)
samples = pd.read_csv(OUTPUT_DIR / "sample_manifest.csv", keep_default_na=False)
flags = pd.read_csv(OUTPUT_DIR / "quality_flags.csv", keep_default_na=False)
summary = json.loads((OUTPUT_DIR / "summary.json").read_text())
print("PlantVillage revision:", summary["plantvillage_revision"])
print("PlantDoc revision:", summary["plantdoc_revision"])
print("Images counted:", summary["images_by_dataset"])
print("Selected samples:", summary["sample"])
DATASET_LABELS = {"plantvillage": "PlantVillage", "plantdoc": "PlantDoc"}
display(counts)


## 2. Class balance by original category

Each bar is the measured total for one category, with its original train/test membership shown separately. These are **file counts**, including any unreadable files; the table above also records readable counts.


In [ ]:
for dataset, display_name in DATASET_LABELS.items():
    subset = counts[counts["dataset"] == dataset]
    pivot = subset.pivot_table(index="category", columns="split", values="file_count",
                               aggfunc="sum", fill_value=0)
    ax = pivot.sort_values(by=list(pivot.columns), ascending=False).plot(
        kind="bar", stacked=True, figsize=(max(11, len(pivot) * .37), 5.5))
    ax.set(title=f"{display_name}: images per source category / split", ylabel="Image files", xlabel="Original category")
    ax.legend(title="Source split")
    plt.xticks(rotation=75, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{dataset.lower()}_class_balance.png", dpi=160)
    plt.show()


## 3. Image dimensions, unreadable files, duplicates, and leaf groups

Different sizes may require resizing, but the resize/crop policy must be checked against real examples. Exact SHA-256 duplicates are confirmed **byte-identical**; separate versions of the same leaf need not have identical hashes. PlantVillage leaf IDs are taken from the author's metadata where resolvable. Any possible leakage needs review before splitting data.


In [ ]:
dimensions = inventory[inventory["readable"].astype(str).str.lower().eq("true")].copy()
dimensions[["width", "height"]] = dimensions[["width", "height"]].apply(pd.to_numeric)
print("Image dimensions:")
display(dimensions.groupby(["dataset", "width", "height"]).size().rename("files")
        .reset_index().sort_values(["dataset", "files"], ascending=[True, False]).reset_index(drop=True))
print("Potential issues:")
if flags.empty:
    print("No automated flags in this run.")
else:
    display(flags.groupby(["dataset", "flag"]).size().rename("flagged_entries").reset_index())
    display(flags)
print("Confirmed byte-identical hash groups:", summary["exact_duplicate_hash_groups"])
print("Cross-partition hash groups:", summary["exact_duplicate_cross_partition_groups"])
print("PlantVillage leaf groups spanning splits:", summary["same_leaf_cross_split_groups"])


## 4. Reproducibly sampled image examples

Samples are selected uniformly, without replacement, per **dataset and original category**, seeded with `551`. The script stores the exact filenames, count, and selection method in `sample_manifest.csv` and `summary.json`. The grid shows up to eight different categories from each dataset. Visually inspect the remaining sampled files as needed; captions use **original** categories. No claim of a representative random sample of all disease severities is implied.


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(18, 6))
for row, (dataset, display_name) in enumerate(DATASET_LABELS.items()):
    chosen = samples[samples["dataset"] == dataset].drop_duplicates("category").head(8)
    for col, (_, item) in enumerate(chosen.iterrows()):
        ax = axes[row, col]
        try:
            with Image.open(DATA_ROOT / item["relative_path"]) as image:
                ax.imshow(ImageOps.exif_transpose(image).convert("RGB"))
        except (OSError, ValueError) as error:
            ax.text(.5, .5, f"Unreadable: {error}", ha="center", va="center", wrap=True)
        ax.set_title(f"{display_name} / {item['category']}", fontsize=8, wrap=True)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "image_examples.png", dpi=160)
plt.show()


## 5. Possible blur: heuristic ranking and manual verification

The edge-variance score uses a grayscale thumbnail (at most 256 px on either side). It is **not a diagnostic blur detector** and changes with texture and background. Low-scoring examples should be compared visually with high-scoring examples *within the same dataset*, rather than automatically discarded. Background clutter, illumination, occlusion, leaf cropping, and suspected label errors all require manual annotation.


In [ ]:
sharpness = samples.copy()
sharpness["edge_variance"] = pd.to_numeric(sharpness["edge_variance"], errors="coerce")
for dataset, display_name in DATASET_LABELS.items():
    ranked = sharpness[sharpness["dataset"] == dataset].sort_values("edge_variance")
    print(f"Lowest edge-variance candidates — {display_name} (inspect visually):")
    display(ranked[["relative_path", "category", "edge_variance"]].head(8).reset_index(drop=True))


## 6. Findings to record after the local run

Fill in the following **only after executing and visually reviewing** the outputs. Keep measured totals separate from the small sampled subset.

| Observation (measured or inspected) | Evidence (table/figure/filenames, count inspected) | Proposed preparation step |
| --- | --- | --- |
| Unreadable or unsupported images | To be checked | Flag/exclude unreadable examples, retaining their paths |
| Dimensions and potential crop loss | To be checked | Consistent resize/normalization; compare leaf visibility before choosing a crop |
| Identical or related images crossing splits | To be checked | Group duplicates and same-leaf photos before final train/validation/test assignment |
| Suspected blur, background clutter, occlusion, or illumination issues | To be checked | Preserve valid hard cases for domain-shift evaluation; consider source-only augmentation only when justified |
| Class imbalance | To be checked | Report per-category counts and use macro-averaged evaluation later |

**Limits:** Current source train/test names are not the project's final experiment partitions. PlantDoc test must not be used to tune preprocessing or model settings. Original category names are not automatically equivalent across datasets. A low edge score is not proof of blur; neither sample inspection nor automated checks validates disease labels. Export selected reviewed charts to `docs/figures/` for Troy's report if needed, with image attribution.
